In [1]:
import xarray as xr
import numpy as np

import os
import glob

In [2]:
directory_m1_qcrad = "/uufs/chpc.utah.edu/common/home/uvu-group1/olson/snow-data/station-data/qcrad1long/*.cdf"
m1_qcrad_list = [np.squeeze(xr.open_dataset(nc)) for nc in sorted(glob.glob(directory_m1_qcrad))]
m1_qcrad = xr.concat(m1_qcrad_list, dim='time')

In [3]:
directory_m1 = "/uufs/chpc.utah.edu/common/home/uvu-group1/olson/snow-data/sebs_SAIL/M1/*.cdf"
m1_list = [np.squeeze(xr.open_dataset(nc)) for nc in sorted(glob.glob(directory_m1))]
m1_sail = xr.concat(m1_list, dim='time')

directory_s3 = "/uufs/chpc.utah.edu/common/home/uvu-group1/olson/snow-data/sebs_SAIL/S3/*.cdf"
s3_list = [np.squeeze(xr.open_dataset(nc)) for nc in sorted(glob.glob(directory_s3))]
s3_sail = xr.concat(s3_list, dim='time')

In [4]:
# Resample once
subset_hourly = m1_qcrad.resample(time='1h').mean().persist()

# Access variables
subset_sw_hourly = subset_hourly['down_short_hemisp']
subset_dif_hourly = subset_hourly['down_short_diffuse_hemisp']
subset_dir_hourly = subset_hourly['short_direct_normal']
subset_best_hourly = subset_hourly['BestEstimate_down_short_hemisp']
subset_zenith_hourly = subset_hourly['zenith']

In [6]:
sub_dirh_hourly = subset_dir_hourly * np.cos(np.deg2rad(subset_zenith_hourly.clip(min=0, max=90))).clip(min=0, max=1)
sub_difh_hourly = subset_dif_hourly * np.cos(np.deg2rad(subset_zenith_hourly.clip(min=0, max=90))).clip(min=0, max=1)

In [7]:
m1_sw_hourly_sail = m1_sail['down_short_hemisp'].resample(time='1h').mean().load()
s3_sw_hourly_sail = s3_sail['down_short_hemisp'].resample(time='1h').mean().load()

In [8]:
# Create folder if it doesn't already exist
save_dir = 'station-data'
os.makedirs(save_dir, exist_ok=True)

# datasets
subset_sw_hourly.to_netcdf(os.path.join(save_dir, 'm1qc_sw.nc'))
subset_dif_hourly.to_netcdf(os.path.join(save_dir, 'm1qc_diff.nc'))
subset_dir_hourly.to_netcdf(os.path.join(save_dir, 'm1qc_dir.nc'))
subset_best_hourly.to_netcdf(os.path.join(save_dir, 'm1qc_best_sw.nc'))
subset_zenith_hourly.to_netcdf(os.path.join(save_dir, 'm1qc_zen.nc'))

sub_dirh_hourly.to_netcdf(os.path.join(save_dir, 'm1qc_dirh.nc'))
sub_difh_hourly.to_netcdf(os.path.join(save_dir, 'm1qc_difh_xx.nc'))

m1_sw_hourly_sail.to_netcdf(os.path.join(save_dir, 'm1_sw.nc'))
s3_sw_hourly_sail.to_netcdf(os.path.join(save_dir, 's3_sw.nc'))